# TP 07 — Fine-tuning DistilBERT for Sentiment Classification

**Course:** Natural Language Processing  
**Session:** 7 / 8

---

## Context

In previous sessions you built models from scratch: feature engineering,
word embeddings, LSTMs, attention. These took minutes to hours to train
and reached ~89% F1 on AG News.

Today you fine-tune **DistilBERT** — a 66M-parameter Transformer pretrained
on 16GB of text — for binary sentiment classification on the **SST-2** dataset.

> **DistilBERT** (Sanh et al., 2019): a 40% smaller, 60% faster distilled version
> of BERT-base that retains 97% of its performance. The key insight:
> knowledge from a large teacher model (BERT) can be transferred to a smaller
> student model via soft probability targets.

**Key question:** what does it mean to fine-tune a pre-trained model?
What changes during training, and what does not?

---

## Roadmap

| Part | Task | New concept |
|------|------|-------------|
| 1 | Load and explore SST-2 | HuggingFace datasets |
| 2 | Tokenise with DistilBERT tokeniser | WordPiece, [CLS], [SEP] |
| 3 | Build the classification model | `TFDistilBertForSequenceClassification` |
| 4 | Train (Keras `.fit`) | frozen vs trainable layers |
| 5 | Evaluate and compare | vs LR baseline from S03 |
| 6 | Analyse predictions | what does the model get wrong? |
| 7 (bonus) | Probing | what does [CLS] encode? |

---


## Setup


In [ ]:
# On Colab: uncomment and run this cell first
# !pip install transformers[tf] datasets scikit-learn matplotlib -q

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

import tensorflow as tf
from transformers import (
    DistilBertTokenizerFast,
    TFDistilBertForSequenceClassification,
    TFAutoModelForSequenceClassification,
)
from datasets import load_dataset

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ── Hyperparameters ───────────────────────────────────────────────────────────
MODEL_NAME   = 'distilbert-base-uncased'
MAX_LEN      = 64     # Maximum token length (SST-2 sentences are short)
BATCH_SIZE   = 32
EPOCHS       = 3
LEARNING_RATE = 2e-5  # Standard for BERT fine-tuning

print(f'TensorFlow: {tf.__version__}')
try:
    import transformers
    print(f'Transformers: {transformers.__version__}')
except:
    print('transformers not installed — run the cell above')
print('Setup OK.')


---
## Part 1 — SST-2 Dataset

**Stanford Sentiment Treebank v2 (SST-2)**:
67K movie review sentences labelled as positive (1) or negative (0).
GLUE benchmark standard for binary sentiment classification.


In [ ]:
# Load from HuggingFace Hub (~5 MB download)
raw_dataset = load_dataset('stanfordnlp/sst2')
print(raw_dataset)


In [ ]:
# Explore
train_data = raw_dataset['train']
val_data   = raw_dataset['validation']

print(f'Train: {len(train_data):,} examples')
print(f'Val  : {len(val_data):,} examples')
print()
print('Sample examples:')
for ex in train_data.select(range(5)):
    label = 'positive' if ex['label'] == 1 else 'negative'
    print(f'  [{label}] {ex["sentence"]}')


### 1.1 — Explore the dataset

Answer the following questions using Python code:

1. What is the class balance (positive vs negative) in the training set?
2. What is the distribution of sentence lengths (in words)? Plot a histogram.
3. Show 3 examples of very short sentences and 3 of very long ones.


In [ ]:
# YOUR CODE HERE
# 1. Class balance

# 2. Length distribution

# 3. Short and long examples


---
## Part 2 — Tokenisation

BERT-family models use **WordPiece** tokenisation, not simple whitespace splitting.

WordPiece:
- Splits rare words into subword units: `'unforgettable'` → `['un', '##forgettable']`
- `##` prefix marks a continuation of the previous word
- Keeps a vocabulary of ~30K tokens (vs millions for word-level)

BERT also adds special tokens:
- `[CLS]` at the start — classification token. Its final embedding is used for classification.
- `[SEP]` at the end — separator token.


### 2.1 — Implement `tokenise_dataset`


In [ ]:
tokeniser = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenise_dataset(
    dataset,
    tokeniser,
    max_len: int,
    batch_size: int = 64,
) -> tf.data.Dataset:
    """
    Tokenise a HuggingFace dataset and return a tf.data.Dataset.

    Steps:
        1. Extract sentences and labels.
        2. Tokenise with truncation and padding to max_len.
        3. Return a tf.data.Dataset of ({'input_ids': ..., 'attention_mask': ...}, labels).

    Parameters
    ----------
    dataset : HuggingFace Dataset
    tokeniser : DistilBertTokenizerFast
    max_len : int
    batch_size : int

    Returns
    -------
    tf.data.Dataset ready for model.fit()

    Notes
    -----
    Use tokeniser() with:
        truncation=True, padding='max_length', max_length=max_len, return_tensors='tf'
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Explore tokenisation on a few examples before building the full dataset
example_sentences = [
    'this movie is great',
    'a forgettable and uninspiring film',
    'the performances are unforgettable',
]
for sent in example_sentences:
    tokens = tokeniser.tokenize(sent)
    print(f'  Input : {sent}')
    print(f'  Tokens: {tokens}')
    print()


In [ ]:
# For speed on CPU: use a subset for training
# On Colab GPU: set TRAIN_SUBSET = None to use the full dataset
TRAIN_SUBSET = 5000  # None = full dataset
VAL_SUBSET   = 872   # full validation set

train_split = train_data.shuffle(seed=RANDOM_STATE)
if TRAIN_SUBSET:
    train_split = train_split.select(range(TRAIN_SUBSET))

train_tf = tokenise_dataset(train_split, tokeniser, MAX_LEN, BATCH_SIZE)
val_tf   = tokenise_dataset(val_data,   tokeniser, MAX_LEN, BATCH_SIZE)

print('Datasets ready.')
print(f'Train subset: {TRAIN_SUBSET or len(train_data):,} examples')
print(f'Val examples: {len(val_data):,}')


---
## Part 3 — Build the Classification Model

`TFDistilBertForSequenceClassification` is a DistilBERT encoder
with a linear classification head on top of the `[CLS]` token.

Architecture:
```
input_ids + attention_mask
    ↓
DistilBERT encoder (6 layers, 768 hidden, 12 heads)
    ↓
[CLS] token embedding (768-dim)
    ↓
Dense(768) → GELU → Dropout
    ↓
Dense(2) → logits
```


### 3.1 — Implement `build_distilbert_model`


In [ ]:
def build_distilbert_model(
    model_name: str,
    learning_rate: float,
    freeze_bert: bool = False,
) -> tf.keras.Model:
    """
    Load and configure TFDistilBertForSequenceClassification.

    Parameters
    ----------
    model_name : str
        HuggingFace model identifier (e.g. 'distilbert-base-uncased').
    learning_rate : float
    freeze_bert : bool
        If True, freeze all DistilBERT weights and only train the head.
        Useful for fast experimentation. If False, fine-tune all layers.

    Returns
    -------
    Compiled tf.keras.Model

    Notes
    -----
    Use TFDistilBertForSequenceClassification.from_pretrained().
    Compile with Adam(learning_rate), SparseCategoricalCrossentropy(from_logits=True),
    and ['accuracy'] metric.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
model = build_distilbert_model(MODEL_NAME, LEARNING_RATE)
model.summary()


---
## Part 4 — Train with Keras `.fit()`

Fine-tuning a pre-trained Transformer is just `model.fit()`.
The HuggingFace TF models are regular Keras models.

> **Fine-tuning vs training from scratch:**  
> DistilBERT was pre-trained on Wikipedia and BooksCorpus for 90 epochs.
> Fine-tuning adapts those representations to our specific task in 3 epochs.
> The pre-trained weights are a starting point — we update *all* of them.

> **Learning rate:** 2e-5 is standard for BERT fine-tuning.
> Higher rates risk catastrophic forgetting of pre-trained representations.


In [ ]:
history = model.fit(
    train_tf,
    validation_data=val_tf,
    epochs=EPOCHS,
    verbose=1,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, metric in zip(axes, ['loss', 'accuracy']):
    ax.plot(history.history[metric],       label='train', color='#56c8ba')
    ax.plot(history.history[f'val_{metric}'], label='val',   color='#e8c47a')
    ax.set_title(metric, color='#e6edf3')
    ax.legend()
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()


---
## Part 5 — Evaluate and Compare

Evaluate DistilBERT on the full SST-2 validation set (872 examples).
Compare to the Logistic Regression baseline from Session 03.


### 5.1 — Implement `evaluate_model`


In [ ]:
def evaluate_model(
    model: tf.keras.Model,
    dataset: tf.data.Dataset,
    label_names: list[str] = ['negative', 'positive'],
) -> tuple[np.ndarray, np.ndarray]:
    """
    Evaluate a classification model on a tf.data.Dataset.

    Parameters
    ----------
    model : tf.keras.Model
    dataset : tf.data.Dataset
        Yields (features_dict, labels) batches.
    label_names : list of str

    Returns
    -------
    y_true : np.ndarray, shape (n,)
    y_pred : np.ndarray, shape (n,)

    Notes
    -----
    The model outputs logits (not probabilities).
    Use tf.argmax to get predicted class indices.
    Print a classification_report.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
y_true, y_pred = evaluate_model(model, val_tf)

# Comparison table
print('\nComparison — SST-2 val accuracy:')
print(f'  Majority baseline : {max(y_true.mean(), 1-y_true.mean()):.3f}')
print(f'  LR + TF-IDF (S03) : ~0.82  (bag-of-words baseline, no context)')
print(f'  DistilBERT (S07)  : {accuracy_score(y_true, y_pred):.3f}')


---
## Part 6 — Error Analysis

High accuracy is not enough. Understanding *what* the model gets wrong
reveals its limitations and biases.


### 6.1 — Implement `predict_sentence`


In [ ]:
def predict_sentence(
    sentence: str,
    model: tf.keras.Model,
    tokeniser,
    max_len: int,
) -> tuple[str, float]:
    """
    Predict the sentiment of a single sentence.

    Parameters
    ----------
    sentence : str
    model : tf.keras.Model
    tokeniser : DistilBertTokenizerFast
    max_len : int

    Returns
    -------
    label : str ('positive' or 'negative')
    confidence : float (probability of the predicted class)

    Notes
    -----
    Convert logits to probabilities with tf.nn.softmax.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Find misclassified examples
val_sentences = val_data['sentence']
val_labels    = val_data['label']

errors = []
for i, (sent, true_label) in enumerate(zip(val_sentences, val_labels)):
    if y_pred[i] != true_label:
        label, conf = predict_sentence(sent, model, tokeniser, MAX_LEN)
        errors.append({
            'sentence': sent,
            'true': 'positive' if true_label == 1 else 'negative',
            'predicted': label,
            'confidence': conf,
        })

print(f'Total errors: {len(errors)} / {len(val_data)} ({100*len(errors)/len(val_data):.1f}%)')
print()
print('High-confidence errors (model was very wrong):')
errors_sorted = sorted(errors, key=lambda x: x['confidence'], reverse=True)
for e in errors_sorted[:5]:
    print(f'  True: {e["true"]:<10} Predicted: {e["predicted"]:<10} Conf: {e["confidence"]:.3f}')
    print(f'  "{e["sentence"]}"')
    print()


**📝 Your analysis**

1. Look at 3–5 high-confidence errors. What do they have in common?
   (negation? irony? domain-specific language? short/ambiguous phrases?)
2. Is the model better at identifying positive or negative sentiment?
   (Look at the classification report from Part 5.)
3. Try these sentences — does the model handle them correctly?
   - `'this is not a good film'` (negation)
   - `'surprisingly not terrible'` (double negation + surprise)
   - `'a masterpiece of mediocrity'` (irony)

*Write your observations here.*


In [ ]:
# Test specific sentences
test_sentences = [
    'this film is great',
    'this film is not great',
    'this is not a good film',
    'surprisingly not terrible',
    'a masterpiece of mediocrity',
    'an absolutely outstanding performance',
    'a complete waste of time',
]
print(f'{"Sentence":<45} {"Prediction":<12} Confidence')
print('-' * 70)
for sent in test_sentences:
    label, conf = predict_sentence(sent, model, tokeniser, MAX_LEN)
    print(f'{sent:<45} {label:<12} {conf:.3f}')


---
## Part 7 (Bonus) — Probing the [CLS] Representation

BERT uses the `[CLS]` token embedding as the input to the classification head.
This embedding is a 768-dimensional vector that encodes the meaning of the sentence.

**Probing question:** do positive and negative sentences cluster separately
in this 768-dimensional space?


In [ ]:
from sklearn.decomposition import PCA

def extract_cls_embeddings(
    sentences: list[str],
    model: tf.keras.Model,
    tokeniser,
    max_len: int,
    batch_size: int = 32,
) -> np.ndarray:
    """
    Extract [CLS] token embeddings for a list of sentences.

    Parameters
    ----------
    sentences : list of str
    model : tf.keras.Model (TFDistilBertForSequenceClassification)
    tokeniser, max_len, batch_size

    Returns
    -------
    np.ndarray, shape (len(sentences), 768)

    Notes
    -----
    To get the hidden state instead of logits, call
    model.distilbert(inputs, output_hidden_states=True).
    The [CLS] embedding is the first token of the last hidden layer:
    hidden_states[-1][:, 0, :]  — shape (batch, 768).
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Use a random sample from the validation set
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(val_data), size=300, replace=False)
sample_sents  = [val_data['sentence'][int(i)] for i in idx]
sample_labels = [val_data['label'][int(i)]    for i in idx]

cls_embs = extract_cls_embeddings(sample_sents, model, tokeniser, MAX_LEN)
print(f'CLS embeddings shape: {cls_embs.shape}')

# PCA to 2D for visualisation
pca = PCA(n_components=2, random_state=RANDOM_STATE)
cls_2d = pca.fit_transform(cls_embs)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#f47067' if l == 0 else '#56c8ba' for l in sample_labels]
ax.scatter(cls_2d[:, 0], cls_2d[:, 1], c=colors, alpha=0.6, s=20)
# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f47067', markersize=8, label='negative'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#56c8ba', markersize=8, label='positive'),
]
ax.legend(handles=legend_elements)
ax.set_title('PCA of DistilBERT [CLS] embeddings (SST-2 val)', color='#e6edf3')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', color='#8b949e')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', color='#8b949e')
ax.set_facecolor('#161b22')
ax.tick_params(colors='#8b949e')
ax.spines[:].set_color('#30363d')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

print('Do positive and negative examples cluster separately? Why or why not?')


---
## Summary


In [ ]:
print('TP-07 — DistilBERT Fine-tuning on SST-2')
print(f'Model: {MODEL_NAME}')
print(f'Dataset: SST-2 ({TRAIN_SUBSET or len(train_data):,} train, {len(val_data):,} val)')
print(f'Hyperparams: MAX_LEN={MAX_LEN}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, LR={LEARNING_RATE}')
print()

from sklearn.metrics import accuracy_score, f1_score
acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average='macro')
print(f'Val Accuracy : {acc:.4f}')
print(f'Val F1 (macro): {f1:.4f}')
print()
print('Next: Session 08 — Advanced topics and course wrap-up')
